# 06 · ONNX export

Export the champion (focal+dice, seed 42) to ONNX, the first step toward edge
deployment. The export is validated for fidelity three ways before it is trusted:
numerical equivalence, exact prediction agreement, and matching test IoU. This is
the same principle as the lossless mask round-trip in notebook 01, applied to the
model: the conversion is verified, not assumed.

## Export

Fixed 1x3x720x960 input, since edge inference runs on the full frame one image at
a time. A fixed shape gives a simpler graph that TensorRT optimizes better than
dynamic axes. The classic TorchScript exporter is used over the newer dynamo
exporter for maturity of the ONNX to TensorRT path; the deprecation warning is
expected and is a conscious choice.

In [2]:
from pathlib import Path

import numpy as np
import torch

from cropweed_seg.model import build_model

ROOT = Path.cwd().parent
device = torch.device("cpu")  # export on CPU for a clean, portable graph

# Load champion
model = build_model(architecture="unet", encoder_name="resnet34")
ckpt = ROOT / "runs" / "focal_dice_s42" / "model.pt"
model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()

# Export. Fixed 960x720 input (full-frame inference, what we deploy).
# NCHW, batch 1 for edge inference.
dummy = torch.randn(1, 3, 720, 960)
out_path = ROOT / "runs" / "focal_dice_s42" / "model.onnx"

torch.onnx.export(
    model,
    dummy,
    str(out_path),
    input_names=["input"],
    output_names=["logits"],
    opset_version=17,
    dynamo=False,            # classic exporter, most stable for TensorRT
    dynamic_axes=None,       # fixed shape: simpler and faster on edge
)
print(f"exported to {out_path}")
print(f"size: {out_path.stat().st_size / 1e6:.1f} MB")

/var/folders/32/b0vbhkmn4w9b2j4s9dlsl0lh0000gn/T/ipykernel_88796/1352645272.py:22: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


exported to /Users/alexmunozestornell/Documents/Personal Projects/Alex's Projects/cropweed-seg/runs/focal_dice_s42/model.onnx
size: 97.7 MB


Exported at 97.7 MB, consistent with 24M float32 parameters. The export ran;
whether it is faithful is what the next checks decide.

## Numerical and prediction equivalence

Run the same random input through PyTorch and ONNX Runtime and compare. The
absolute difference should be tiny (floating-point noise from different operation
implementations), and the argmax predictions, which are what actually matter,
must agree exactly.

In [4]:
import onnxruntime as ort

# Same random input through both, compare outputs
test_input = torch.randn(1, 3, 720, 960)

with torch.no_grad():
    torch_out = model(test_input).numpy()

session = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])
onnx_out = session.run(["logits"], {"input": test_input.numpy()})[0]

# Numerical equivalence: should match to a small tolerance
max_diff = np.abs(torch_out - onnx_out).max()
print(f"output shapes: torch {torch_out.shape}, onnx {onnx_out.shape}")
print(f"max absolute difference: {max_diff:.2e}")

# Argmax (the actual predictions) must match exactly
torch_pred = torch_out.argmax(axis=1)
onnx_pred = onnx_out.argmax(axis=1)
pred_match = (torch_pred == onnx_pred).mean()
print(f"prediction agreement: {pred_match:.4%}")

output shapes: torch (1, 3, 720, 960), onnx (1, 3, 720, 960)
max absolute difference: 1.28e-05
prediction agreement: 100.0000%


Max difference 1.28e-05, pure floating-point noise, and prediction agreement
100%. The ONNX model produces the same masks as PyTorch, pixel for pixel.

## Test IoU

The final check: the ONNX model must give the same per-class IoU on the whole
test split, not just on one random input. This confirms the export preserves the
project's metric, reusing the same ConfusionMatrixMetric as everywhere else.

In [5]:
import onnxruntime as ort

from cropweed_seg.dataset import PeanutDataset
from cropweed_seg.metrics import ConfusionMatrixMetric
from cropweed_seg.transforms import build_transforms

test_ds = PeanutDataset(ROOT, "test", build_transforms("test"))
session = ort.InferenceSession(str(out_path), providers=["CPUExecutionProvider"])

metric = ConfusionMatrixMetric()
for idx in range(len(test_ds)):
    image, mask = test_ds[idx]
    onnx_logits = session.run(["logits"], {"input": image.unsqueeze(0).numpy()})[0]
    pred = torch.from_numpy(onnx_logits).argmax(dim=1)  # (1, H, W)
    metric.update(pred, mask.unsqueeze(0))

onnx_iou = metric.compute()
print("ONNX model on test split:")
for k, v in onnx_iou.items():
    print(f"  {k:<16} {v:.4f}")
print("\nPyTorch was: weed 0.6695, mIoU 0.8500")

ONNX model on test split:
  iou_background   0.9711
  iou_crop         0.9096
  iou_weed         0.6695
  miou             0.8500

PyTorch was: weed 0.6695, mIoU 0.8500


## Result

The ONNX model matches PyTorch exactly on test: weed 0.6695, mIoU 0.8500,
identical to four decimals. The export is faithful by all three checks. The model
is deployable, and this artifact stands on its own regardless of the later
hardware steps. Quantization comes next, in notebook 07.